In [114]:
import numpy  as np 
import pandas as pd 


In [115]:
data =  pd.read_csv('train.csv')

In [116]:
data.columns

Index(['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType',
       'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature',
       'Weather'],
      dtype='object')

In [117]:
data

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy
...,...,...,...,...,...,...,...,...,...,...,...
77294,77294,qp0d4n,49,2:0,0.067203,Residential,1,Not Allowed,No,11.501664,Rainy
77295,77295,qp0d4q,49,2:0,0.022859,Residential,3,Allowed,Yes,14.715254,Foggy
77296,77296,qp0d4w,49,2:0,0.141342,Residential,3,Allowed,Yes,19.678860,Sunny
77297,77297,qp0dhw,49,2:0,0.087574,Residential,1,Not Allowed,No,22.573958,Sunny


In [118]:
data = data.drop(columns=['Index'])


In [119]:
for col  in data.columns : 
    print(col,"-> ",data[col].isna().sum()," dtype ",data[col].dtype)

geohash ->  0  dtype  object
day ->  0  dtype  int64
timestamp ->  0  dtype  object
demand ->  0  dtype  float64
RoadType ->  600  dtype  object
NumberofLanes ->  0  dtype  int64
LargeVehicles ->  0  dtype  object
Landmarks ->  0  dtype  object
Temperature ->  2495  dtype  float64
Weather ->  797  dtype  object


In [120]:
data

,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy
...,...,...,...,...,...,...,...,...,...,...
77294,qp0d4n,49,2:0,0.067203,Residential,1,Not Allowed,No,11.501664,Rainy
77295,qp0d4q,49,2:0,0.022859,Residential,3,Allowed,Yes,14.715254,Foggy
77296,qp0d4w,49,2:0,0.141342,Residential,3,Allowed,Yes,19.678860,Sunny
77297,qp0dhw,49,2:0,0.087574,Residential,1,Not Allowed,No,22.573958,Sunny


In [121]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

cat_cols = ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks']

for col in cat_cols:
    le = LabelEncoder()

    data[col] = le.fit_transform(data[col].astype(str))

    label_encoders[col] = le

In [122]:
data

,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,qp02z1,48,0:0,0.048804,3,1,1,0,NaN,4
1,qp02zt,48,0:0,0.118507,1,3,0,1,31.104565,3
2,qp08bj,48,0:0,0.027132,1,1,1,0,25.919267,3
3,qp08gt,48,0:0,0.003272,1,1,1,0,NaN,1
4,qp02zq,48,0:0,0.010819,1,1,1,0,10.803667,1
...,...,...,...,...,...,...,...,...,...,...
77294,qp0d4n,49,2:0,0.067203,1,1,1,0,11.501664,1
77295,qp0d4q,49,2:0,0.022859,1,3,0,1,14.715254,0
77296,qp0d4w,49,2:0,0.141342,1,3,0,1,19.678860,3
77297,qp0dhw,49,2:0,0.087574,1,1,1,0,22.573958,3


In [123]:
for col  in data.columns:
    print(col ,"-> ",data[col].isna().sum())

geohash ->  0
day ->  0
timestamp ->  0
demand ->  0
RoadType ->  0
NumberofLanes ->  0
LargeVehicles ->  0
Landmarks ->  0
Temperature ->  2495
Weather ->  0


In [124]:
temp_median = data['Temperature'].median()

data['Temperature'] = data['Temperature'].fillna(temp_median)

In [125]:
for col  in data.columns:
    print(col ,"-> ",data[col].isna().sum())

geohash ->  0
day ->  0
timestamp ->  0
demand ->  0
RoadType ->  0
NumberofLanes ->  0
LargeVehicles ->  0
Landmarks ->  0
Temperature ->  0
Weather ->  0


In [126]:
print(data['Temperature'].describe())


count    77299.000000
mean        16.404619
std          7.240083
min        -14.935097
25%         11.624189
50%         16.382587
75%         21.115592
max         48.251433
Name: Temperature, dtype: float64


In [127]:
full_train_data =  data

In [128]:
# =========================
# Frequency Encoding
# =========================

geo_count = full_train_data['geohash'].value_counts()

full_train_data['geo_freq'] = (
    full_train_data['geohash']
    .map(geo_count)
)

# =========================
# Timestamp Features
# =========================

full_train_data[['hour', 'minute']] = (
    full_train_data['timestamp']
    .str.split(':', expand=True)
    .astype(int)
)

# =========================
# Cyclic Features
# =========================

full_train_data['hour_sin'] = np.sin(2 * np.pi * full_train_data['hour'] / 24)
full_train_data['hour_cos'] = np.cos(2 * np.pi * full_train_data['hour'] / 24)

full_train_data['minute_sin'] = np.sin(2 * np.pi * full_train_data['minute'] / 60)
full_train_data['minute_cos'] = np.cos(2 * np.pi * full_train_data['minute'] / 60)

# =========================
# Interaction Features
# =========================

full_train_data['lanes_temp'] = (
    full_train_data['NumberofLanes'] * full_train_data['Temperature']
)

full_train_data['temp_hour'] = (
    full_train_data['Temperature'] * full_train_data['hour']
)

# =========================
# Target Encoding
# =========================

global_mean = full_train_data['demand'].mean()

geo_mean = (
    full_train_data
    .groupby('geohash')['demand']
    .mean()
)

full_train_data['geohash_target'] = (
    full_train_data['geohash']
    .map(geo_mean)
    .fillna(global_mean)
)

# =========================
# Time Block
# =========================

def get_period(h):
    if 5 <= h <= 11:
        return "morning"
    elif 11 <= h <= 16:
        return "afternoon"
    else:
        return "night"

full_train_data['time_block'] = (
    full_train_data['hour']
    .apply(get_period)
    .astype('category')
)

# =========================
# Road Broadness
# =========================

full_train_data['road_broadness'] = (
    full_train_data['RoadType'].astype(str)
    + "_"
    + full_train_data['NumberofLanes'].astype(str)
).astype('category')

# =========================
# Large Vehicles Time
# =========================

full_train_data['large_vehicles_time'] = (
    full_train_data['LargeVehicles'].astype(str)
    + "_"
    + full_train_data['hour'].astype(str)
).astype('category')

# =========================
# Peak Time
# =========================

full_train_data['is_peak_time'] = (
    full_train_data['hour']
    .isin([7, 8, 9, 17, 18, 19])
    .astype(int)
)

# =========================
# Weather Hour
# =========================

full_train_data['weather_hour'] = (
    full_train_data['Weather'].astype(str)
    + "_"
    + full_train_data['hour'].astype(str)
).astype('category')

# =========================
# Road Hour
# =========================

full_train_data['road_hour'] = (
    full_train_data['RoadType'].astype(str)
    + "_"
    + full_train_data['hour'].astype(str)
).astype('category')

# =========================
# Geo Hour
# =========================

full_train_data['geo_hour'] = (
    full_train_data['geohash'].astype(str)
    + "_"
    + full_train_data['hour'].astype(str)
).astype('category')

# =========================
# Frequency Features
# =========================

weather_count = full_train_data['Weather'].value_counts()

full_train_data['weather_freq'] = (
    full_train_data['Weather']
    .map(weather_count)
)

road_count = full_train_data['RoadType'].value_counts()

full_train_data['road_freq'] = (
    full_train_data['RoadType']
    .map(road_count)
)

# =========================
# NEW FEATURES
# =========================

full_train_data['geo_road'] = (
    full_train_data['geohash'].astype(str)
    + "_"
    + full_train_data['RoadType'].astype(str)
).astype('category')

full_train_data['geo_target_hour'] = (
    full_train_data['geohash_target']
    * full_train_data['hour']
)

full_train_data['geo_target_freq'] = (
    full_train_data['geohash_target']
    * full_train_data['geo_freq']
)

full_train_data['geo_target_road'] = (
    full_train_data['geohash_target']
    * (full_train_data['RoadType'] + 1)
)

full_train_data['geo_target_temp'] = (
    full_train_data['geohash_target']
    * full_train_data['Temperature']
)


geo_hour_count = (
    full_train_data['geo_hour']
    .value_counts()
)

full_train_data['geo_hour_freq'] = (
    full_train_data['geo_hour']
    .map(geo_hour_count)
)

# road_pressure

full_train_data['road_pressure'] = (
    full_train_data['LargeVehicles']
    * full_train_data['RoadType']
    * full_train_data['geo_freq']
)

# road_density

full_train_data['road_density'] = (
    full_train_data['geo_freq']
    / (full_train_data['NumberofLanes'] + 1)
)

# road_capacity_time

full_train_data['road_capacity_time'] = (
    full_train_data['NumberofLanes']
    * (full_train_data['hour'] + 1)
    * (full_train_data['RoadType'] + 1)
)
# =========================
# Final Cleanup
# =========================

full_train_data['geohash'] = (
    full_train_data['geohash']
    .astype('category')
)

full_train_data = full_train_data.drop(
    columns=['timestamp']
)

full_train_data_y = full_train_data['demand']

full_train_data = full_train_data.drop(
    columns=['demand']
)



In [129]:
full_train_data

,geohash,day,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,geo_freq,hour,...,road_freq,geo_road,geo_target_hour,geo_target_freq,geo_target_road,geo_target_temp,geo_hour_freq,road_pressure,road_density,road_capacity_time
0,qp02z1,48,3,1,1,0,16.382587,4,33,0,...,600,qp02z1_3,0.000000,1.321572,0.160191,0.656084,7,99,16.50,4
1,qp02zt,48,1,3,0,1,31.104565,3,89,0,...,69230,qp02zt_1,0.000000,18.580183,0.417532,6.493579,8,0,22.25,6
2,qp08bj,48,1,1,1,0,25.919267,3,67,0,...,69230,qp08bj_1,0.000000,8.571384,0.255862,3.315880,6,67,33.50,2
3,qp08gt,48,1,1,1,0,16.382587,1,42,0,...,69230,qp08gt_1,0.000000,0.604002,0.028762,0.235598,5,42,21.00,2
4,qp02zq,48,1,1,1,0,10.803667,1,36,0,...,69230,qp02zq_1,0.000000,1.054789,0.058599,0.316544,3,36,18.00,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77294,qp0d4n,49,1,1,1,0,11.501664,1,68,2,...,69230,qp0d4n_1,0.053020,1.802689,0.053020,0.304911,5,68,34.00,6
77295,qp0d4q,49,1,3,0,1,14.715254,0,74,2,...,69230,qp0d4q_1,0.278417,10.301414,0.278417,2.048485,3,0,18.50,18
77296,qp0d4w,49,1,3,0,1,19.678860,3,101,2,...,69230,qp0d4w_1,0.226439,11.435175,0.226439,2.228032,5,0,25.25,18
77297,qp0dhw,49,1,1,1,0,22.573958,3,88,2,...,69230,qp0dhw_1,0.150149,6.606566,0.150149,1.694731,5,88,44.00,6


In [130]:
full_train_data.columns

Index(['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles',
       'Landmarks', 'Temperature', 'Weather', 'geo_freq', 'hour', 'minute',
       'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'lanes_temp',
       'temp_hour', 'geohash_target', 'time_block', 'road_broadness',
       'large_vehicles_time', 'is_peak_time', 'weather_hour', 'road_hour',
       'geo_hour', 'weather_freq', 'road_freq', 'geo_road', 'geo_target_hour',
       'geo_target_freq', 'geo_target_road', 'geo_target_temp',
       'geo_hour_freq', 'road_pressure', 'road_density', 'road_capacity_time'],
      dtype='object')

In [131]:
test_data =  pd.read_csv('test.csv')

In [132]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          41778 non-null  int64  
 1   geohash        41778 non-null  object 
 2   day            41778 non-null  int64  
 3   timestamp      41778 non-null  object 
 4   RoadType       41454 non-null  object 
 5   NumberofLanes  41778 non-null  int64  
 6   LargeVehicles  41778 non-null  object 
 7   Landmarks      41778 non-null  object 
 8   Temperature    40429 non-null  float64
 9   Weather        41347 non-null  object 
dtypes: float64(1), int64(3), object(6)
memory usage: 3.2+ MB


In [133]:
# =========================
# Fill Missing Values
# =========================

test_data['Temperature'] = test_data['Temperature'].fillna(temp_median)

# =========================
# Apply Saved Label Encoders
# =========================

cat_cols = ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks']

for col in cat_cols:
    test_data[col] = label_encoders[col].transform(
        test_data[col].astype(str)
    )

# =========================
# Frequency Encoding
# =========================

test_data['geo_freq'] = (
    test_data['geohash']
    .map(geo_count)
    .fillna(0)
)


# =========================
# Timestamp Features
# =========================

test_data[['hour', 'minute']] = (
    test_data['timestamp']
    .str.split(':', expand=True)
    .astype(int)
)

# =========================
# Cyclic Features
# =========================

test_data['hour_sin'] = np.sin(2 * np.pi * test_data['hour'] / 24)
test_data['hour_cos'] = np.cos(2 * np.pi * test_data['hour'] / 24)

test_data['minute_sin'] = np.sin(2 * np.pi * test_data['minute'] / 60)
test_data['minute_cos'] = np.cos(2 * np.pi * test_data['minute'] / 60)

# =========================
# Interaction Features
# =========================

test_data['lanes_temp'] = (
    test_data['NumberofLanes'] * test_data['Temperature']
)

test_data['temp_hour'] = (
    test_data['Temperature'] * test_data['hour']
)

# =========================
# Target Encoding
# =========================

test_data['geohash_target'] = (
    test_data['geohash']
    .map(geo_mean)
    .fillna(global_mean)
)

# =========================
# Time Block
# =========================

def get_period(h):
    if 5 <= h <= 11:
        return "morning"
    elif 11 <= h <= 16:
        return "afternoon"
    else:
        return "night"

test_data['time_block'] = (
    test_data['hour']
    .apply(get_period)
    .astype('category')
)
test_data['geo_target_temp'] = (
    test_data['geohash_target']
    * test_data['Temperature']
)

# =========================
# Road Broadness
# =========================

test_data['road_broadness'] = (
    test_data['RoadType'].astype(str)
    + "_"
    + test_data['NumberofLanes'].astype(str)
).astype('category')

# =========================
# Large Vehicles Time
# =========================

test_data['large_vehicles_time'] = (
    test_data['LargeVehicles'].astype(str)
    + "_"
    + test_data['hour'].astype(str)
).astype('category')

# =========================
# Peak Time
# =========================

test_data['is_peak_time'] = (
    test_data['hour']
    .isin([7, 8, 9, 17, 18, 19])
    .astype(int)
)

# =========================
# Weather Hour
# =========================

test_data['weather_hour'] = (
    test_data['Weather'].astype(str)
    + "_"
    + test_data['hour'].astype(str)
).astype('category')

# =========================
# Road Hour
# =========================

test_data['road_hour'] = (
    test_data['RoadType'].astype(str)
    + "_"
    + test_data['hour'].astype(str)
).astype('category')

test_data['geohash'] = test_data['geohash'].astype('category')

# =========================
# Drop Timestamp
# =========================

test_data = test_data.drop(columns=['timestamp'])

# =========================
# Weather Frequency Encoding
# =========================

test_data['weather_freq'] = (
    test_data['Weather']
    .map(weather_count)
    .fillna(0)
)

# =========================
# RoadType Frequency Encoding
# =========================

test_data['road_freq'] = (
    test_data['RoadType']
    .map(road_count)
    .fillna(0)
)

# =========================
# Geo Hour
# =========================

test_data['geo_hour'] = (
    test_data['geohash'].astype(str)
    + "_"
    + test_data['hour'].astype(str)
).astype('category')

# =========================
# Geo Road
# =========================

test_data['geo_road'] = (
    test_data['geohash'].astype(str)
    + "_"
    + test_data['RoadType'].astype(str)
).astype('category')

# =========================
# New Numeric Features
# =========================

test_data['geo_target_hour'] = (
    test_data['geohash_target']
    * test_data['hour']
)

test_data['geo_target_freq'] = (
    test_data['geohash_target']
    * test_data['geo_freq']
)

test_data['geo_target_road'] = (
    test_data['geohash_target']
    * (test_data['RoadType'] + 1)
)

test_data['geo_hour_freq'] = (
    test_data['geo_hour']
    .map(geo_hour_count)
    .fillna(0)
)

test_data['road_pressure'] = (
    test_data['LargeVehicles']
    * test_data['RoadType']
    * test_data['geo_freq']
)

# =========================
# road_density
# =========================

test_data['road_density'] = (
    test_data['geo_freq']
    / (test_data['NumberofLanes'] + 1)
)

# =========================
# road_capacity_time
# =========================

test_data['road_capacity_time'] = (
    test_data['NumberofLanes']
    * (test_data['hour'] + 1)
    * (test_data['RoadType'] + 1)
)

test_data['geohash'] = test_data['geohash'].astype('category')
test_data['geo_hour'] = test_data['geo_hour'].astype('category')
test_data['geo_road'] = test_data['geo_road'].astype('category')

In [134]:
full_train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 36 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   geohash              77299 non-null  category
 1   day                  77299 non-null  int64   
 2   RoadType             77299 non-null  int64   
 3   NumberofLanes        77299 non-null  int64   
 4   LargeVehicles        77299 non-null  int64   
 5   Landmarks            77299 non-null  int64   
 6   Temperature          77299 non-null  float64 
 7   Weather              77299 non-null  int64   
 8   geo_freq             77299 non-null  int64   
 9   hour                 77299 non-null  int64   
 10  minute               77299 non-null  int64   
 11  hour_sin             77299 non-null  float64 
 12  hour_cos             77299 non-null  float64 
 13  minute_sin           77299 non-null  float64 
 14  minute_cos           77299 non-null  float64 
 15  lanes_temp         

In [135]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 37 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   Index                41778 non-null  int64   
 1   geohash              41778 non-null  category
 2   day                  41778 non-null  int64   
 3   RoadType             41778 non-null  int64   
 4   NumberofLanes        41778 non-null  int64   
 5   LargeVehicles        41778 non-null  int64   
 6   Landmarks            41778 non-null  int64   
 7   Temperature          41778 non-null  float64 
 8   Weather              41778 non-null  int64   
 9   geo_freq             41778 non-null  float64 
 10  hour                 41778 non-null  int64   
 11  minute               41778 non-null  int64   
 12  hour_sin             41778 non-null  float64 
 13  hour_cos             41778 non-null  float64 
 14  minute_sin           41778 non-null  float64 
 15  minute_cos         

In [136]:
print(full_train_data.dtypes[full_train_data.dtypes == 'category'])
print(test_data.dtypes[test_data.dtypes == 'category'])

geohash                category
time_block             category
road_broadness         category
large_vehicles_time    category
weather_hour           category
road_hour              category
geo_hour               category
geo_road               category
dtype: object
geohash                category
time_block             category
road_broadness         category
large_vehicles_time    category
weather_hour           category
road_hour              category
geo_hour               category
geo_road               category
dtype: object


In [137]:
full_train_data.columns

Index(['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles',
       'Landmarks', 'Temperature', 'Weather', 'geo_freq', 'hour', 'minute',
       'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'lanes_temp',
       'temp_hour', 'geohash_target', 'time_block', 'road_broadness',
       'large_vehicles_time', 'is_peak_time', 'weather_hour', 'road_hour',
       'geo_hour', 'weather_freq', 'road_freq', 'geo_road', 'geo_target_hour',
       'geo_target_freq', 'geo_target_road', 'geo_target_temp',
       'geo_hour_freq', 'road_pressure', 'road_density', 'road_capacity_time'],
      dtype='object')

In [138]:
# same columns
test_final = test_data[full_train_data.columns]

print((full_train_data.columns == test_final.columns).all())

# missing values
print(test_final.isnull().sum().sum())

True
0


In [139]:
print(full_train_data.columns.tolist())
print(test_final.columns.tolist())

['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'geo_freq', 'hour', 'minute', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'lanes_temp', 'temp_hour', 'geohash_target', 'time_block', 'road_broadness', 'large_vehicles_time', 'is_peak_time', 'weather_hour', 'road_hour', 'geo_hour', 'weather_freq', 'road_freq', 'geo_road', 'geo_target_hour', 'geo_target_freq', 'geo_target_road', 'geo_target_temp', 'geo_hour_freq', 'road_pressure', 'road_density', 'road_capacity_time']
['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'geo_freq', 'hour', 'minute', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'lanes_temp', 'temp_hour', 'geohash_target', 'time_block', 'road_broadness', 'large_vehicles_time', 'is_peak_time', 'weather_hour', 'road_hour', 'geo_hour', 'weather_freq', 'road_freq', 'geo_road', 'geo_target_hour', 'geo_target_freq', 'geo_target_road', 'geo_target_temp', 'geo_hou

In [140]:
test_data

for col in test_data.columns:
    print(f"{col} | dtype {test_data[col].dtype} |")

Index | dtype int64 |
geohash | dtype category |
day | dtype int64 |
RoadType | dtype int64 |
NumberofLanes | dtype int64 |
LargeVehicles | dtype int64 |
Landmarks | dtype int64 |
Temperature | dtype float64 |
Weather | dtype int64 |
geo_freq | dtype float64 |
hour | dtype int64 |
minute | dtype int64 |
hour_sin | dtype float64 |
hour_cos | dtype float64 |
minute_sin | dtype float64 |
minute_cos | dtype float64 |
lanes_temp | dtype float64 |
temp_hour | dtype float64 |
geohash_target | dtype float64 |
time_block | dtype category |
geo_target_temp | dtype float64 |
road_broadness | dtype category |
large_vehicles_time | dtype category |
is_peak_time | dtype int64 |
weather_hour | dtype category |
road_hour | dtype category |
weather_freq | dtype int64 |
road_freq | dtype int64 |
geo_hour | dtype category |
geo_road | dtype category |
geo_target_hour | dtype float64 |
geo_target_freq | dtype float64 |
geo_target_road | dtype float64 |
geo_hour_freq | dtype float64 |
road_pressure | dtype

In [141]:
print(full_train_data.columns)
for col in full_train_data.columns :
    print(f"{col} -> {full_train_data[col].dtype} ")

Index(['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles',
       'Landmarks', 'Temperature', 'Weather', 'geo_freq', 'hour', 'minute',
       'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'lanes_temp',
       'temp_hour', 'geohash_target', 'time_block', 'road_broadness',
       'large_vehicles_time', 'is_peak_time', 'weather_hour', 'road_hour',
       'geo_hour', 'weather_freq', 'road_freq', 'geo_road', 'geo_target_hour',
       'geo_target_freq', 'geo_target_road', 'geo_target_temp',
       'geo_hour_freq', 'road_pressure', 'road_density', 'road_capacity_time'],
      dtype='object')
geohash -> category 
day -> int64 
RoadType -> int64 
NumberofLanes -> int64 
LargeVehicles -> int64 
Landmarks -> int64 
Temperature -> float64 
Weather -> int64 
geo_freq -> int64 
hour -> int64 
minute -> int64 
hour_sin -> float64 
hour_cos -> float64 
minute_sin -> float64 
minute_cos -> float64 
lanes_temp -> float64 
temp_hour -> float64 
geohash_target -> float64 
time_block -> c

In [142]:
#models intialization &  training 
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor 


cat_features_cat = [
     'time_block',
    'road_broadness',
    'large_vehicles_time',
    'road_hour',
    'weather_hour',
    'geohash',
    'geo_hour',
    'geo_road'
]

cat_features_lgb = [
    # 'time_block',
    # 'road_broadness',
    # 'large_vehicles_time',
    # 'road_hour',
    # 'weather_hour',
]

cat_boost_model =  CatBoostRegressor(
    iterations=5000,
    learning_rate=0.01,
    depth =8,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed= 42,
    task_type='GPU',
    devices='0',
    verbose= False
    )

cat_boost_model.fit(full_train_data,full_train_data_y,
          cat_features= cat_features_cat,
          )

lgb_model =  LGBMRegressor(
    n_estimators= 5000,
    learning_rate= 0.03,
    num_leaves = 63,
    min_child_samples=20,
    subsample=0.8,
    max_depth=-1,
    random_state= 42,
    n_jobs = -1,
    verbose = -1
    )

lgb_model.fit(full_train_data,full_train_data_y,
              categorical_feature=cat_features_lgb,
              )

,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.03
,n_estimators,5000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [143]:
cat_pred =  cat_boost_model.predict(test_final)
lgb_pred =  lgb_model.predict(test_final)

ensemble =  cat_pred * 0.7 + 0.3* lgb_pred 


In [144]:
print("Min :", ensemble.min())
print("Max :", ensemble.max())
print("Mean:", ensemble.mean())

print("NaNs:", np.isnan(ensemble).sum())

Min : 0.0036114898829326804
Max : 1.0282631730686895
Mean: 0.12483857683937014
NaNs: 0


In [145]:
submission = pd.DataFrame({
    "Index": test_data["Index"],
    "demand": ensemble
})

submission.head()

,Index,demand
0,0,0.049738
1,1,0.017895
2,2,0.024086
3,3,0.028731
4,4,0.049628


In [146]:
print(submission.shape)
print(submission.head())

(41778, 2)
   Index    demand
0      0  0.049738
1      1  0.017895
2      2  0.024086
3      3  0.028731
4      4  0.049628


In [148]:
submission.to_csv("submission12.csv", index=False)